# ReVA Video Analysis

Reads the 18-video source-stratified sample in `reports/video_sample_analysis.json`, fetches only those MP4s through the revision-pinned Hub cache, measures FPS/duration/resolution/frame counts, displays sampled frames, and records failures. Results describe this sample only, not the full dataset.

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from preprocessing.hf_access import fetch_dataset_file
from preprocessing.video_loader import get_video_metadata, read_frames
from preprocessing.frame_sampling import uniform_indices
manifest=json.loads(Path("manifests/reva_manifest.json").read_text(encoding="utf-8"))
repo,revision=manifest["repo_id"],manifest["revision"]
sample_report=json.loads(Path("reports/video_sample_analysis.json").read_text(encoding="utf-8"))
sample_paths=[item["video_path"] for item in sample_report["sampled_videos"]]
print(sample_report["sampling_design"])

In [ ]:
records=[]; decoded={}; failures=[]
for rel in sample_paths:
    try:
        path=fetch_dataset_file(repo,rel,revision=revision)
        meta=get_video_metadata(path)
        ids=uniform_indices(meta["frame_count"],8).tolist()
        frames=read_frames(path,ids)
        records.append({"video_path":rel,"file_bytes":path.stat().st_size,**meta,"decoded_frames":len(frames)})
        decoded[rel]=(frames,ids)
    except Exception as exc:
        failures.append({"video_path":rel,"error":str(exc)})
video_df=pd.DataFrame(records)
print("sample success",len(records),"of",len(sample_paths),"failures",failures)
video_df

In [ ]:
example_items=list(decoded.items())[:6]
fig,axes=plt.subplots(2,3,figsize=(13,7))
for ax,(rel,(frames,indices)) in zip(axes.flat,example_items):
    ax.imshow(frames[0]); ax.set_title(f"{rel.split('/')[-1]}\nframe {indices[0]}"); ax.axis("off")
for ax in axes.flat[len(example_items):]: ax.axis("off")
plt.tight_layout()

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(13,3.5))
for column,ax in zip(["fps","duration_seconds","frame_count"],axes):
    ax.hist(video_df[column].dropna(),bins=min(8,max(1,len(video_df))))
    ax.set_title(f"Sample {column}")
plt.tight_layout()
print("resolution counts",video_df.groupby(["width","height"]).size().to_dict())
print("corrupted/unreadable in selected sample",len(failures),"of",len(sample_paths))